
# 02: Reading the results

Get the numbers out of a fit, one by one or all in a table.


## Introduction

In the first tutorial, ``report`` did everything at once: fit, summary and figure.

Here we look at what a fit gives back, and how to use it: read one number, check that the
fit is good, keep all the numbers in a table, and choose the part of the curve to describe.



In [1]:
# Import what we need
from evoparam import fit_response, gen_response

## Fitting without drawing

``fit_response`` does only the fit: no figure, no summary. It is the one to use when you
have many curves, or when you only want the numbers.

We use the same curve as in the first tutorial: a sharp bump at 25 ms and a broad one at
150 ms.



In [2]:
# Make the example curve
times, signal = gen_response([[25, -150, 10], [150, -80, 80]], seed=3)

# Fit the model, without drawing
fit = fit_response(times, signal, preset="ccep")

``fit`` now holds everything evoparam found. ``print(fit)`` still shows the summary at any
time.



In [3]:
print(fit)

                    evoparam - RESPONSE MODEL

                     fit window: 9 to 500 ms
          background: exponential, time constant 35.4 ms

                      2 component(s) found:
            label        latency     amplitude     width
            N1           25.0 ms    -152.1 µV     9.5 ms
            N2          151.9 ms     -77.4 µV    79.2 ms

           fit quality: R2 = 0.94, noise level = 7.4 µV
  (latency: time of the peak; width: full width at half maximum)


## Reading the bumps

The bumps are in ``fit.components``, a list in order of time. Each bump has the three
numbers of the first tutorial:

- ``latency``: when the bump peaks;
- ``amplitude``: its height above the background, in µV;
- ``fwhm``: its width at half its height ("full width at half maximum").

<div class="alert alert-info"><h4>Note</h4>

Inside evoparam, times are in **seconds**: a latency of 0.025 means 25 ms. Multiply
by 1000 to read them in milliseconds.

</div>



In [4]:
# Go through the bumps, one line each
for bump in fit.components:
    print(bump.label, round(bump.latency * 1000, 1), "ms,", round(bump.amplitude, 1), "µV")

N1 25.0 ms, -152.1 µV
N2 151.9 ms, -77.4 µV


To keep one number, take the bump by its place in the list: ``[0]`` is the first one.



In [5]:
# The first bump, and its latency in ms
n1 = fit.components[0]
n1_latency_ms = n1.latency * 1000
print(round(n1_latency_ms, 1))

25.0


Each bump also has a few more numbers, all measured on the bump alone:

- ``onset`` and ``offset``: when the bump starts and ends (seconds);
- ``area``: the surface of the bump, the coloured area of the first tutorial
  (µV × seconds);
- ``rise_time`` and ``decay_time``: how long it takes to reach its peak, and to come
  back.



In [6]:
# When the first bump starts and ends, in ms
print(round(n1.onset * 1000, 1), round(n1.offset * 1000, 1))

16.3 33.6


## Reading the background

The background is in ``fit.background``. Here it is a curve that fades away, and the
number to read is its **time constant**: how long it takes to lose about two thirds of its
height. The smaller it is, the faster it fades.



In [7]:
# The time constant of the background, in ms
print(round(fit.background["tau"] * 1000, 1))

35.4


## Is the fit good?

Before using the numbers, check that the model really follows the curve. Two numbers in
``fit.gof`` ("goodness of fit") say it:

- ``r2``: how much of the curve the model explains, from 0 (nothing) to 1 (everything);
- ``rmse``: the typical distance between the curve and the model, in µV. It should be
  close to the level of the noise.



In [8]:
print(round(fit.gof.r2, 2), round(fit.gof.rmse, 1))

0.94 7.1


<div class="alert alert-info"><h4>Note</h4>

Here R² is about 0.94: the model explains 94 % of the curve, and what is left is the
noise we put in. A low R² means the model missed something: look at the figure
(``report``) before trusting the numbers.

</div>



## All the numbers in a table

``to_dataframe`` puts every number of every bump in a table, one row per bump. This table
can be sorted, plotted or saved like any spreadsheet (it needs the ``pandas`` package,
already there in most notebooks).



In [9]:
# One row per bump; show a few columns
table = fit.to_dataframe()
table[["label", "latency", "amplitude", "fwhm", "area"]]

,label,latency,amplitude,fwhm,area
0,N1,0.024951,-152.057403,0.009520,-1.491724
1,N2,0.151916,-77.413645,0.079193,-6.317850


To keep it, save it as a file that any spreadsheet program opens:

```python
table.to_csv("my_results.csv")
```


## Choosing the part of the curve to describe

The preset decides which part of the curve is described: for ``"ccep"``, from 9 to 500 ms
after the stimulus. To look at another part, give a ``window``, in seconds.

Here we keep only the first 100 ms: the broad bump at 150 ms is left out, and only the
sharp one is described.



In [10]:
# Describe only 9 to 100 ms after the stimulus
early = fit_response(times, signal, preset="ccep", window=(0.009, 0.1))
print(early)

                    evoparam - RESPONSE MODEL

                     fit window: 9 to 100 ms
          background: exponential, time constant 50.0 ms

                      1 component(s) found:
            label        latency     amplitude     width
            N1           24.9 ms    -152.4 µV     9.5 ms

           fit quality: R2 = 0.97, noise level = 7.4 µV
  (latency: time of the peak; width: full width at half maximum)


<div class="alert alert-info"><h4>Note</h4>

The preset matters too: it holds the settings made for one kind of recording.
``"ccep"`` is for responses to electrical pulses in the brain; ``"erp"`` is for
responses recorded on the scalp (EEG), ``"sep"`` for touch, ``"tep"`` for magnetic
pulses (TMS), and there are a few more.

</div>



## Conclusion

A fit gives back its bumps (``fit.components``), its background (``fit.background``) and
how good it is (``fit.gof``), and ``to_dataframe`` puts everything in one table. With
``window`` and ``preset``, you choose what is described.

